# Bảng điểm micro và macro, đọc thẳng từ Drive

Notebook **chỉ đọc**. Không clone mã, không train, không xoá gì. Chạy vài giây.

## Vì sao cần

`docs/BANG_DIEM.md` ghi **macro** — trung bình theo người, là số chính thức của
đồ án theo `docs/PROTOCOL.md` mục 4.

Nhưng bài báo gốc công bố **micro** — trung bình toàn bộ phiên đo. Hai thước cho
hai con số khác nhau, nên khi đặt cạnh nhau phải nói rõ đang dùng thước nào.

Cả hai đều được ghi sẵn vào `summary.csv` ở mọi lần chạy
(`run_cv.py` và `run_final_test.py`), nên chỉ cần đọc ra, không phải chạy lại gì.

## Cách dùng

Chạy mục 1 và 2, rồi bấm mục nào cần. Mỗi mục độc lập.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 2. Hàm đọc

Tệp nén là tích luỹ, nên đọc hết rồi gộp theo `run_id`; trùng thì giữ bản của tệp mới nhất.

In [ ]:
import csv, glob, io, os, re, subprocess

DRIVE = "/content/drive/MyDrive/mobivital/"

def doc_summary(mau):
    """Đọc summary.csv BÊN TRONG các tệp nén khớp mẫu. Không giải nén ra đĩa.

    Trả về dict {run_id: dòng}. Tệp nén là tích luỹ nên đọc hết rồi gộp,
    giữ bản của tệp mới nhất khi trùng run_id.
    """
    rows = {}
    tep = sorted(glob.glob(DRIVE + mau), key=os.path.getmtime)
    for f in tep:
        ds = subprocess.run(["unzip", "-Z1", f], capture_output=True, text=True).stdout
        for path in [x for x in ds.split() if x.endswith("summary.csv")]:
            raw = subprocess.run(["unzip", "-p", f, path],
                                 capture_output=True, text=True).stdout
            for r in csv.DictReader(io.StringIO(raw)):
                rows[r["run_id"]] = r
    print("%d tệp nén khớp '%s'  ->  %d dòng" % (len(tep), mau, len(rows)))
    return rows

def la_ket_qua_du(r):
    """Dòng đại diện cho một cấu hình đã chạy xong.

    run_cv.py ghi dòng fold TONG khi đủ 4 fold; run_final_test.py không có
    cột fold vì mỗi lần chạy chỉ một dòng.
    """
    return r.get("fold", "") in ("TONG", "", None)

def bang(rows, loc=""):
    """In run_id kèm micro và macro."""
    chon = [r for r in rows.values() if la_ket_qua_du(r) and loc in r["run_id"]]
    print()
    print("  %-58s %10s %10s" % ("cấu hình", "micro", "macro"))
    print("  " + "-" * 80)
    for r in sorted(chon, key=lambda r: r["run_id"]):
        # Cắt ĐẦU chứ không cắt đuôi: seed và alpha nằm ở cuối run_id,
        # mất chúng là không phân biệt được các dòng với nhau.
        ten = r["run_id"]
        if len(ten) > 58:
            ten = "…" + ten[-57:]
        print("  %-58s %10s %10s"
              % (ten, r.get("score_micro", "—")[:8], r.get("score_macro", "—")[:8]))
    print("  " + "-" * 80)
    print("  %d cấu hình" % len(chon))

## 3. TN2 — tầm nhìn, hai bề rộng kênh đặt cạnh nhau

In [ ]:
RF = {3: 61, 5: 121, 7: 181, 9: 241, 11: 301, 13: 361}

rows = doc_summary("tn2_rf*.zip")

theo_kenh = {}
for r in rows.values():
    if not la_ket_qua_du(r):
        continue
    m = re.search(r"_c(\d+)_k(\d+)_", r["run_id"])
    if m:
        theo_kenh[(int(m.group(1)), int(m.group(2)))] = r

print()
print("  tầm nhìn  kernel          c64                       c192")
print("                       micro      macro         micro      macro")
print("  " + "-" * 64)
for k in sorted({k for _, k in theo_kenh}):
    o = []
    for ch in (64, 192):
        r = theo_kenh.get((ch, k))
        o.append("%9s %10s" % (r.get("score_micro", "—")[:8], r.get("score_macro", "—")[:8])
                 if r else "%9s %10s" % ("—", "—"))
    print("  %-9d %-6d %s   %s" % (RF.get(k, 0), k, o[0], o[1]))
print("  " + "-" * 64)

## 4. TN3 — quét hàm loss

In [ ]:
rows = doc_summary("tn3*.zip")
bang(rows)

## 5. TN4 — test cuối trên tập kiểm tra độc lập

In [ ]:
rows = doc_summary("tn4*.zip")
bang(rows)

## 6. TN1 — so kiến trúc

In [ ]:
rows = doc_summary("tn1*.zip")
bang(rows)

## 7. Xem tất cả, lọc theo chuỗi bất kỳ

Đổi hai dòng dưới. Ví dụ `loc="_c64_"` chỉ xem 64 kênh.

In [ ]:
rows = doc_summary("*.zip")
bang(rows, loc="")